# Basic Attention

Basic Attention Unit:

In [33]:
import torch
import torch.nn as nn

class SelfAttention_v2(nn.Module):
    def __init__(self, d_in, d_out, qkv_bias=False):
        super().__init__()
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key   = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)

    def forward(self, x):
        keys = self.W_key(x)
        queries = self.W_query(x)
        values = self.W_value(x)
        
        attn_scores = queries @ keys.T
        attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)

        context_vec = attn_weights @ values
        return context_vec


Causal Attention with Dropout:

In [34]:
class CausalAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, qkv_bias=False):
        super().__init__()
        self.d_out = d_out
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key   = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.dropout = nn.Dropout(dropout)
        
        # Kausale Maske im Buffer registrieren
        self.register_buffer(
            "mask", 
            torch.triu(torch.ones(context_length, context_length), diagonal=1)
        )

    def forward(self, x):
        b, num_tokens, d_in = x.shape
        keys = self.W_key(x)
        queries = self.W_query(x)
        values = self.W_value(x)

        # MatMul für Batches: (b, num_tokens, d_out) @ (b, d_out, num_tokens)
        attn_scores = queries @ keys.transpose(1, 2)
        
        # Maskierung mit -inf
        attn_scores.masked_fill_(
            self.mask.bool()[:num_tokens, :num_tokens], -torch.inf
        )
        
        attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)
        attn_weights = self.dropout(attn_weights)
        
        context_vec = attn_weights @ values
        return context_vec

Multi-head Attention:

In [41]:
class MultiHeadAttention(nn.Module):
    #Einstellbar: num_heads -> Anzahl der Attention-Heads, Größe der Matrizen(k) -> d_in,d_out (Eingabedimension, Ausgabedimension), Temperatur = temperature
    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False, temperature=1.0):
        super().__init__()
        assert (d_out % num_heads == 0), "d_out must be divisible by num_heads"
        
        self.d_out = d_out
        self.num_heads = num_heads
        self.head_dim = d_out // num_heads  # Projektionsdimension pro Head
        self.temperature = temperature # Temperatur einstellbar
        
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key   = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.out_proj = nn.Linear(d_out, d_out)  # Lineare Ausgabeprojektion
        self.dropout = nn.Dropout(dropout)
        
        self.register_buffer(
            "mask",
            torch.triu(torch.ones(context_length, context_length), diagonal=1)
        )

    def forward(self, x):
        b, num_tokens, d_in = x.shape
        
        keys = self.W_key(x)
        queries = self.W_query(x)
        values = self.W_value(x)

        # Umformen für Multi-Head: (b, num_tokens, num_heads, head_dim)
        keys = keys.view(b, num_tokens, self.num_heads, self.head_dim)
        values = values.view(b, num_tokens, self.num_heads, self.head_dim)
        queries = queries.view(b, num_tokens, self.num_heads, self.head_dim)

        # Transponieren: (b, num_heads, num_tokens, head_dim)
        keys = keys.transpose(1, 2)
        queries = queries.transpose(1, 2)
        values = values.transpose(1, 2)

        # Scaled Dot-Product Attention pro Head
        attn_scores = queries @ keys.transpose(2, 3)
        mask_bool = self.mask.bool()[:num_tokens, :num_tokens]
        attn_scores.masked_fill_(mask_bool, -torch.inf)

        scaled_scores = attn_scores / (self.head_dim**0.5 * self.temperature) # Temperatur einstellbar

        #attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)
        attn_weights = torch.softmax(scaled_scores, dim=-1)
        attn_weights = self.dropout(attn_weights)

        context_vec = (attn_weights @ values).transpose(1, 2)
        # Combine heads, where self.d_out = self.num_heads * self.head_dim
        context_vec = context_vec.contiguous().view(b, num_tokens, self.d_out)
        context_vec = self.out_proj(context_vec)
        
        return context_vec

Tests (mit variablen Werten für Temperatur, Anzahl Attention-Heads, Größe der Matrizen):

In [48]:
import torch

# 1. Set seed for exact reproducibility
torch.manual_seed(420)

# 2. Fetch a single batch from DataLoader
input_batch, _ = next(iter(dataloader))
seq_len = input_batch.shape[1]

print("=" * 65)
print("TEST 1: Base Configuration (d_in=16, d_out=16, num_heads=2, temp=1.0)")
print("=" * 65)

# Create embeddings (token + positional)
d_in1, d_out1 = 16, 16
token_emb1 = torch.nn.Embedding(200000, d_in1)
pos_emb1 = torch.nn.Embedding(seq_len, d_in1)
batch1 = token_emb1(input_batch) + pos_emb1(torch.arange(seq_len))

# Unpack tensor shape
batch_size, context_length, d_in = batch1.shape

# Instantiate and run MHA
mha1 = MultiHeadAttention(
    d_in=d_in1,
    d_out=d_out1,
    context_length=context_length,
    dropout=0.0,
    num_heads=2,
    temperature=1.0
)
context_vecs1 = mha1(batch1)

print("context_vecs1.shape:", context_vecs1.shape)
print("First 4 values of token 0:\n", context_vecs1[0, 0, :4].detach().numpy().round(4))


print("\n" + "=" * 65)
print("TEST 2: Larger Matrix & Low Temperature (d_out=64, num_heads=8, temp=0.5)")
print("=" * 65)

# Create embeddings with larger feature dimension (64)
d_in2, d_out2 = 64, 64
token_emb2 = torch.nn.Embedding(200000, d_in2)
pos_emb2 = torch.nn.Embedding(seq_len, d_in2)
batch2 = token_emb2(input_batch) + pos_emb2(torch.arange(seq_len))

mha2 = MultiHeadAttention(
    d_in=d_in2,
    d_out=d_out2,
    context_length=context_length,
    dropout=0.0,
    num_heads=8,
    temperature=0.5  # Sharper attention focus
)
context_vecs2 = mha2(batch2)

print("context_vecs2.shape:", context_vecs2.shape)
print("First 4 values of token 0:\n", context_vecs2[0, 0, :4].detach().numpy().round(4))


print("\n" + "=" * 65)
print("TEST 3: High Temperature Effect (d_out=64, num_heads=8, temp=2.0)")
print("=" * 65)

mha3 = MultiHeadAttention(
    d_in=d_in2,
    d_out=d_out2,
    context_length=context_length,
    dropout=0.0,
    num_heads=8,
    temperature=2.0  # Broader/softer attention distribution
)
context_vecs3 = mha3(batch2)

print("context_vecs3.shape:", context_vecs3.shape)
print("First 4 values of token 0:\n", context_vecs3[0, 0, :4].detach().numpy().round(4))

TEST 1: Base Configuration (d_in=16, d_out=16, num_heads=2, temp=1.0)
context_vecs1.shape: torch.Size([8, 4, 16])
First 4 values of token 0:
 [-0.0968  0.3006 -0.1342 -0.9093]

TEST 2: Larger Matrix & Low Temperature (d_out=64, num_heads=8, temp=0.5)
context_vecs2.shape: torch.Size([8, 4, 64])
First 4 values of token 0:
 [ 0.0907 -0.1543 -0.6081 -0.1652]

TEST 3: High Temperature Effect (d_out=64, num_heads=8, temp=2.0)
context_vecs3.shape: torch.Size([8, 4, 64])
First 4 values of token 0:
 [-0.8086 -0.723   0.112   0.4602]
